In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/data-sprint-prompt-based-classififcation/merged_test_30.csv
/kaggle/input/competitions/data-sprint-prompt-based-classififcation/merged_train_70.csv


# Malicious Prompt Detection - Hackathon Submission

**Participation Details:**
* **Full Name:** Shourya Varshney
* **College Name:** GLA University, Mathura
* **Phone Number:** 7247870237
* **Email Address:** varshneyshourya19@gmail.com
* **Google Drive Link (Model, Output & Metrics):** [Paste your public Google Drive link here]
* 
### **Project Overview & The Engineering Journey**
Sharing my final approach for this track. Getting to this final model wasn't a straight line. I ran into a few classic deep-learning hurdles along the way, and I'm documenting them here to explain the reasoning behind my final architecture.

**1. The Data Mess & The "Accuracy Trap"**
The training dataset had heavily misaligned columns, with malicious and safe prompts scattered across different fields. After cleaning that up, I noticed a massive imbalance: ~297k safe prompts vs ~14k malicious ones. 
I initially tried a 1:3 sampling ratio to mimic a real-world environment. The result was a 0.00 F1 score. The model fell into the "Accuracy Trap"—it realized it could just guess "Safe" every single time, achieve 75% accuracy for free, and completely ignore the malicious patterns.

**2. The Hardware Limit (The DeBERTa Crash)**
To catch complex adversarial patterns, I tried upgrading the engine to `microsoft/deberta-v3-base`. While it's mathematically superior, it is notoriously unstable on Kaggle's T4 GPUs. I hit a gradient collapse (`Validation Loss: NaN`), meaning the weights exploded during training. Fighting an unstable model during a time-constrained sprint is a bad idea, so I pivoted.

**3. The Final Strategy: 50/50 Balance + DistilBERT**
I went back to stability and speed. 
* **The Data:** I enforced a strict 50/50 split (approx. 14k safe and 14k malicious). This forces the model to actually learn the adversarial patterns because there is no majority class to fall back on. 
* **The Engine:** I used `distilbert-base-uncased`. It is lightweight, mathematically stable, and fast. It trained flawlessly in about 15 minutes, converging beautifully and securing a highly competitive F1 score without overfitting.

In [2]:
import pandas as pd
train_df = pd.read_csv('/kaggle/input/competitions/data-sprint-prompt-based-classififcation/merged_train_70.csv')   # Update the path!
test_df = pd.read_csv('/kaggle/input/competitions/data-sprint-prompt-based-classififcation/merged_test_30.csv')
print(train_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 357477 entries, 0 to 357476
Data columns (total 17 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Unnamed: 0    45590 non-null   float64
 1   idx           45590 non-null   float64
 2   Prompt        73121 non-null   object 
 3   Length        45409 non-null   float64
 4   Perplexity    43832 non-null   float64
 5   embedding     46522 non-null   object 
 6   isMalicious   27531 non-null   float64
 7   id            283244 non-null  float64
 8   qid1          283244 non-null  float64
 9   qid2          283244 non-null  float64
 10  question1     283243 non-null  object 
 11  question2     283242 non-null  object 
 12  is_duplicate  283244 non-null  float64
 13  category      1112 non-null    object 
 14  base_class    1112 non-null    object 
 15  text          1112 non-null    object 
 16  Unnamed: 0.1  31681 non-null   float64
dtypes: float64(10), object(7)
memory usage: 46.4+ MB

### Data Wrangling & Alignment
Extracting the text and labels from the misaligned columns to create a clean, unified dataframe.

In [3]:
import pandas as pd

# extracting the insights see
df_labeled = train_df[['Prompt', 'isMalicious']].dropna()
df_labeled.rename(columns={'Prompt': 'text', 'isMalicious': 'label'}, inplace=True)

# Extracting the questions
df_quora = train_df[['question1']].dropna()
df_quora['label'] = 0.0
df_quora.rename(columns={'question1': 'text'}, inplace=True)

# Extract the "text" column from the Malignant dataset
df_malignant = train_df[['text', 'base_class']].dropna()
df_malignant['label'] = df_malignant['base_class'].apply(lambda x: 1.0 if str(x).lower() != 'safe' else 0.0)
df_malignant.drop(columns=['base_class'], inplace=True)

# Combine everything into one master
clean_train = pd.concat([df_labeled, df_quora, df_malignant], ignore_index=True)

# Shuffling the dataset.
clean_train = clean_train.sample(frac=1, random_state=42).reset_index(drop=True)
clean_train['label'] = clean_train['label'].astype(int)

# Printing the final shape
print(clean_train.info())
print("\nLabel Balance:\n", clean_train['label'].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311886 entries, 0 to 311885
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    311886 non-null  object
 1   label   311886 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 4.8+ MB
None

Label Balance:
 label
0    297008
1     14878
Name: count, dtype: int64


### Handling the Imbalance (50/50 Split)
Undersampling the safe prompts to exactly match the malicious prompts. This prevents the model from blindly guessing the majority class.

In [4]:
# Separate the classes
df_safe = clean_train[clean_train['label'] == 0]
df_malicious = clean_train[clean_train['label'] == 1]

# Undersample the safe class to exactly match the malicious class
df_safe_balanced = df_safe.sample(n=len(df_malicious), random_state=42)

# Combine them back together into a final, balanced dataset
final_train = pd.concat([df_safe_balanced, df_malicious], ignore_index=True)

# Shuffle the data one last time
final_train = final_train.sample(frac=1, random_state=42).reset_index(drop=True)

print("--- FINAL BALANCED DATASET ---")
print(final_train['label'].value_counts())
print(f"Total rows for fast training: {len(final_train)}")

--- FINAL BALANCED DATASET ---
label
1    14878
0    14878
Name: count, dtype: int64
Total rows for fast training: 29756


### Transformer Fine-Tuning
Loading the DistilBERT model and the standard Trainer. I'm using 2 epochs, which is the sweet spot for this dataset size to prevent overfitting.

In [5]:
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score

print("Starting Transformer Setup...")

# 1. Split the balanced data (80% for training, 20% for testing our F1 score)
train_split = final_train.sample(frac=0.8, random_state=42)
val_split = final_train.drop(train_split.index)

# Convert Pandas DataFrames to Hugging Face Datasets
train_ds = Dataset.from_pandas(train_split)
val_ds = Dataset.from_pandas(val_split)

# 2. Load the Tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenization function (keeps max length to 128 to train incredibly fast tonight)
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

print("Tokenizing data (this takes about 30 seconds)...")
train_tokenized = train_ds.map(tokenize_function, batched=True)
val_tokenized = val_ds.map(tokenize_function, batched=True)

# 3. Define the F1 Metric for the Judges
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "f1": f1_score(labels, predictions),
        "accuracy": accuracy_score(labels, predictions)
    }

# 4. Load the Pre-trained Model Structure (Allowed by T&Cs)
print("Downloading DistilBERT...")
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 5. Set up Training Arguments for a Fast Sprint
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",    # Check F1 score after every epoch
    learning_rate=2e-5,
    per_device_train_batch_size=32, # Maximize GPU usage
    per_device_eval_batch_size=32,
    num_train_epochs=2,             # 2 epochs is plenty for 30k rows
    weight_decay=0.01,
    report_to="none"                # Turn off external logging to prevent Kaggle errors
)

# 6. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
)

print("Starting Training! Grab a coffee, this will take about 15 minutes...")
# 7. TRAIN!
trainer.train()

Starting Transformer Setup...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=744, training_loss=0.2277780912255728, metrics={'train_runtime': 308.6197, 'train_samples_per_second': 154.268, 'train_steps_per_second': 2.411, 'total_flos': 1576693212503040.0, 'train_loss': 0.2277780912255728, 'epoch': 2.0})

### Output Generation & Packaging
Running the test set through the trained model, generating the required CSV, and automatically saving the validation metrics and zipped model for the final submission.

In [6]:
import pandas as pd
import numpy as np
from datasets import Dataset
import shutil
import os

print("1. Preparing Test Data...")
# Combine the scattered text columns into exactly 'text'
test_df['text'] = test_df['Prompt'].fillna(test_df['question1']).fillna(test_df['text'])

# Ensure the text is strings and keep the id
test_clean = test_df[['id', 'text']].copy()

# Fix the ID column (using the .to_series() fix)
test_clean['id'] = test_clean['id'].fillna(test_clean.index.to_series()).astype(int)
test_clean['text'] = test_clean['text'].astype(str)

print("2. Tokenizing Test Data...")
test_ds = Dataset.from_pandas(test_clean)
test_tokenized = test_ds.map(tokenize_function, batched=True)

print("3. Generating Predictions (This takes about 2-3 minutes)...")
# Run the test data through your trained model
raw_predictions = trainer.predict(test_tokenized)

# Convert the raw probabilities into 0 or 1 labels
final_labels = np.argmax(raw_predictions.predictions, axis=-1)

print("4. Saving Submission File...")
# Create the exact format required: id, label
submission_df = pd.DataFrame({
    'id': test_clean['id'],
    'label': final_labels
})

# Save to CSV
submission_filename = 'submission.csv'
submission_df.to_csv(submission_filename, index=False)
print(f"SUCCESS! Predictions saved to {submission_filename}")

print("5. Saving and Zipping the Model...")
# Define absolute Kaggle paths
model_dir = "/kaggle/working/my_trained_model"
zip_path = "/kaggle/working/trained_model_submission"

# Save weights and tokenizer
trainer.save_model(model_dir)
tokenizer.save_pretrained(model_dir)

# Compress the folder into a .zip file
shutil.make_archive(zip_path, 'zip', model_dir)

print(f"SUCCESS! The model zip file has been created as {zip_path}.zip")
print("\n--- ALL TASKS COMPLETE! ---")
print("Refresh your Kaggle output panel to download submission.csv and trained_model_submission.zip")

1. Preparing Test Data...
2. Tokenizing Test Data...


Map:   0%|          | 0/153205 [00:00<?, ? examples/s]

3. Generating Predictions (This takes about 2-3 minutes)...


4. Saving Submission File...
SUCCESS! Predictions saved to submission.csv
5. Saving and Zipping the Model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SUCCESS! The model zip file has been created as /kaggle/working/trained_model_submission.zip

--- ALL TASKS COMPLETE! ---
Refresh your Kaggle output panel to download submission.csv and trained_model_submission.zip


# **For more help i also create you a Output text file for your help**

In [7]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, f1_score

print("Generating the Output File for Judges...")

# Calculate the metrics using your validation data 
# (Since the hidden test data doesn't have the answers for us to check)
val_predictions_raw = trainer.predict(val_tokenized)
val_preds = np.argmax(val_predictions_raw.predictions, axis=-1)
val_true_labels = val_split['label'].tolist()

cm = confusion_matrix(val_true_labels, val_preds)
acc = accuracy_score(val_true_labels, val_preds)
prec = precision_score(val_true_labels, val_preds)
f1 = f1_score(val_true_labels, val_preds)

# Create the dedicated text file
with open('DataSprint_Output_Metrics.txt', 'w') as f:
    f.write("--- DATASPRINT OUTPUT METRICS ---\n")
    f.write("Note: Metrics calculated on the 20% validation split.\n\n")
    f.write(f"1. F1 Score:  {f1:.4f}\n")
    f.write(f"2. Accuracy:  {acc:.4f}\n")
    f.write(f"3. Precision: {prec:.4f}\n\n")
    f.write(f"4. Confusion Matrix:\n{cm}\n")

print("SUCCESS! Metrics saved to 'DataSprint_Output_Metrics.txt'")

Generating the Output File for Judges...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


SUCCESS! Metrics saved to 'DataSprint_Output_Metrics.txt'
